# Can we predict whether an OkCupid user smokes?

A binary classification study on the 2012 OkCupid profile extract.

|  |  |
|---|---|
| **Data** | `profiles.csv` — 59,946 public OkCupid profiles from the San Francisco Bay Area, 2012 (distributed by Codecademy) |
| **Target** | `smokes`, collapsed into a binary "ever smoker" label |
| **Features** | `drugs`, `body_type`, `age`, `drinks`, plus a religion ablation |
| **Models** | Logistic regression and random forest, scored on a held-out 20% test set |

This is the modelling half of the project. The interactive companion is the
Streamlit app in the repository root.

### The short version

Most of the interesting result here is negative, and it is worth stating plainly.

| Model | Accuracy | Precision (smoker) | Recall (smoker) | F1 (smoker) |
|---|---|---|---|---|
| Baseline — always "does not smoke" | 0.806 | 0.000 | 0.000 | 0.000 |
| Logistic regression | 0.820 | 0.618 | 0.184 | 0.283 |
| Logistic regression, `class_weight="balanced"` | 0.709 | 0.351 | 0.592 | 0.441 |
| Random forest, unlimited depth | 0.714 | 0.345 | 0.535 | 0.420 |
| Random forest, `max_depth=5` | 0.751 | 0.392 | 0.520 | 0.447 |

Scored on a held-out test set of 10,887 profiles, 19.4% of them smokers.

Three things this notebook establishes:

1. **80.6% of the people who answer the smoking question do not smoke.** So the
   plain logistic regression's 82.0% accuracy is not a result — it is the base rate
   plus noise. That model catches 18% of actual smokers.
2. **Making the two classes cost the same fixes the right thing and wrecks the
   headline number.** Balancing the weights drops accuracy to 70.9% while lifting
   smoker recall from 0.18 to 0.59. If the goal is finding smokers rather than
   scoring well, that trade is worth making, and it is invisible if you only ever
   report accuracy.
3. **The signal is almost entirely drug use and drinking.** `drugs_often` has the
   largest coefficient by a wide margin (2.07), then `drugs_sometimes` (1.38) and
   `drinks_very often` (1.02). Age is close to irrelevant (-0.04) and no body type
   exceeds 0.36.

The overall finding is that smoking is predictable only in a weak sense: the best
model reaches 0.45 F1 on the minority class, and cross-validation shows the choice
of tree depth barely matters as long as it is capped at all.

## 1. Load the data

The full extract is 151 MB, which is over GitHub's file limit, so it is not committed — see `DATA.md`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 1000)

candidates = [
    Path("data/full/profiles.csv"),
    Path("../data/full/profiles.csv"),
    Path("data/sample_profiles.csv"),
    Path("../data/sample_profiles.csv"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find profiles.csv — see DATA.md")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH} -> {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded ..\data\full\profiles.csv -> 59,946 rows x 31 columns


,age,body_type,diet,drinks,drugs,education,essay0,essay1,essay2,essay3,...,location,offspring,orientation,pets,religion,sex,sign,smokes,speaks,status
0,22,a little extra,strictly anything,socially,never,working on college/university,about me:<br />\n<br />\ni would love to think...,currently working as an international agent fo...,making people laugh.<br />\nranting about a go...,"the way i look. i am a six foot half asian, ha...",...,"south san francisco, california","doesn&rsquo;t have kids, but might want them",straight,likes dogs and likes cats,agnosticism and very serious about it,m,gemini,sometimes,english,single
1,35,average,mostly other,often,sometimes,working on space camp,i am a chef: this is what that means.<br />\n1...,dedicating everyday to being an unbelievable b...,being silly. having ridiculous amonts of fun w...,NaN,...,"oakland, california","doesn&rsquo;t have kids, but might want them",straight,likes dogs and likes cats,agnosticism but not too serious about it,m,cancer,no,"english (fluently), spanish (poorly), french (...",single
2,38,thin,anything,socially,NaN,graduated from masters program,"i'm not ashamed of much, but writing public te...","i make nerdy software for musicians, artists, ...",improvising in different contexts. alternating...,my large jaw and large glasses are the physica...,...,"san francisco, california",NaN,straight,has cats,NaN,m,pisces but it doesn&rsquo;t matter,no,"english, french, c++",available
3,23,thin,vegetarian,socially,NaN,working on college/university,i work in a library and go to school. . .,reading things written by old dead people,playing synthesizers and organizing books acco...,socially awkward but i do my best,...,"berkeley, california",doesn&rsquo;t want kids,straight,likes cats,NaN,m,pisces,no,"english, german (poorly)",single
4,29,athletic,NaN,socially,never,graduated from college/university,hey how's it going? currently vague on the pro...,work work work work + play,creating imagery to look at:<br />\nhttp://bag...,i smile a lot and my inquisitive nature,...,"san francisco, california",NaN,straight,likes dogs and likes cats,NaN,m,aquarius,no,english,single


## 2. Feature engineering

Two columns need tidying before they are usable. `location` carries a city and a
state, and `religion` bundles a belief together with how seriously the person
takes it.

In [2]:
# City, kept only where the dataset holds enough profiles for it to mean
# something: anything with fewer than 11 entries collapses into "other".
df["city"] = df["location"].str.split(", ").str[0]
city_counts = df["city"].value_counts()

df["city"] = df["city"].where(df["city"].map(city_counts) >= 11, "other")

df["city"].value_counts().head(10)

city
san francisco    31064
oakland           7214
berkeley          4212
san mateo         1331
palo alto         1064
alameda            910
san rafael         755
hayward            747
emeryville         738
redwood city       693
Name: count, dtype: int64

In [3]:
# "agnosticism and laughing about it" -> ("agnosticism", "laughing about it")
df["religion_main"] = df["religion"].str.split(r" but | and ", regex=True).str[0]
df["religion_attitude"] = df["religion"].str.split(r" but | and ", regex=True).str[1]

df["religion_main"].value_counts(dropna=False).head(12)

religion_main
NaN             20226
agnosticism      8812
other            7743
atheism          6985
christianity     5787
catholicism      4758
judaism          3098
buddhism         1948
hinduism          450
islam             139
Name: count, dtype: int64

## 3. Defining the target

`smokes` takes five values plus missing. Rather than model all five, the question
here is the simpler and more useful one: **does this person smoke at all?**

Before building anything it is worth knowing the base rate, because that single
number decides whether any accuracy figure that follows is impressive or
meaningless.

In [4]:
df["smokes"].value_counts(dropna=False)

smokes
no                43896
NaN                5512
sometimes          3787
when drinking      3040
yes                2231
trying to quit     1480
Name: count, dtype: int64

In [5]:
known = df["smokes"].dropna()
non_smoker_share = (known == "no").mean()

print(f"Profiles answering the smoking question: {len(known):,} of {len(df):,} ({len(known) / len(df):.1%})")
print(f"Of those, {non_smoker_share:.1%} do not smoke")
print(f"=> always answering 'does not smoke' already scores {non_smoker_share:.3f} accuracy")

Profiles answering the smoking question: 54,434 of 59,946 (90.8%)
Of those, 80.6% do not smoke
=> always answering 'does not smoke' already scores 0.806 accuracy


That 80.6% is the whole story of this notebook, so it is worth restating before
any model is fitted: **a model that always answers "does not smoke" scores 0.806
accuracy without learning anything about anything.** Any accuracy figure near 0.81
is therefore a failure wearing a costume.

The target is also pinned down here. `trying to quit` is counted as a smoker, so
the label means "ever smoked" rather than "currently smokes". That single choice
moves 1,480 rows, and a stricter definition would change every number below.

In [6]:
# "Ever smoker": anyone reporting any smoking at all, including people who are
# currently trying to quit.
smoke_map = {
    "no": 0,
    "sometimes": 1,
    "when drinking": 1,
    "yes": 1,
    "trying to quit": 1,
}

df["smokes_binary"] = df["smokes"].map(smoke_map)
df["smokes_binary"].value_counts(dropna=False)

smokes_binary
0.0    43896
1.0    10538
NaN     5512
Name: count, dtype: int64

In [7]:
print(df["drinks"].value_counts(dropna=False))
print()
print(df["drugs"].value_counts(dropna=False))

drinks
socially       41780
rarely          5957
often           5164
not at all      3267
NaN             2985
very often       471
desperately      322
Name: count, dtype: int64

drugs
never        37724
NaN          14080
sometimes     7732
often          410
Name: count, dtype: int64


## 4. Building the feature matrix

Four features go in: two lifestyle habits, one body descriptor and age. The
categorical ones are one-hot encoded with `drop_first=True`, which drops one
level per column as the reference category.

Rows missing a `smokes` answer are dropped — they cannot train or score a model —
but missing values in the *features* are left alone. `pd.get_dummies` encodes a
missing category as all-zeros, which makes it indistinguishable from the dropped
reference level. The cell below measures how often that happens, because it is
easy to overlook.

In [8]:
df = df.dropna(subset=["smokes_binary"])

features = ["drugs", "body_type", "age", "drinks"]
X = df[features]
y = df["smokes_binary"].astype(int)

X = pd.get_dummies(X, columns=["drugs", "body_type", "drinks"], drop_first=True)

print("X:", X.shape, "| y:", y.shape)
X.head()

X: (54434, 19) | y: (54434,)


,age,drugs_often,drugs_sometimes,body_type_athletic,body_type_average,body_type_curvy,body_type_fit,body_type_full figured,body_type_jacked,body_type_overweight,body_type_rather not say,body_type_skinny,body_type_thin,body_type_used up,drinks_not at all,drinks_often,drinks_rarely,drinks_socially,drinks_very often
0,22,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,35,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,38,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
3,23,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
4,29,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


In [9]:
missing = df[["age", "drugs", "body_type", "drinks", "smokes_binary"]].isna().sum()
print(missing.sort_values(ascending=False))
print()
print("Rows that land on all-zeros for a one-hot block (missing, or the dropped reference):")
for column in ["drugs", "body_type", "drinks"]:
    block = [name for name in X.columns if name.startswith(column + "_")]
    print(f"  {column:10s} {int((X[block].sum(axis=1) == 0).sum()):>6,}")

drugs            11317
body_type         4297
drinks            1032
age                  0
smokes_binary        0
dtype: int64

Rows that land on all-zeros for a one-hot block (missing, or the dropped reference):
  drugs      46,729
  body_type   6,778
  drinks      1,321


Two things about this matrix are worth noticing before modelling it.

**Missing features are silently folded into the reference category.** There are
11,317 rows with no drug answer, but 46,729 rows encode as all-zeros for the `drugs`
block — because `drop_first=True` also drops "never", the most common answer. The
model therefore cannot distinguish "reports never using drugs" from "did not answer
the question". With 14,080 missing drug values in total, that is not a rounding
error, and it is the weakest point of this feature set.

**Body type is mostly ballast.** It supplies twelve of the nineteen columns and, as
the coefficients later show, moves the prediction far less than either habit does.
It is kept because it is cheap here, not because it has earned its place.

## 5. Train / test split

The split is **stratified** so the smoker rate is identical in both halves. With a
19% minority class, an unlucky unstratified split would move the base rate and
quietly change every score below.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train {X_train.shape[0]:,} rows | test {X_test.shape[0]:,} rows")
print(f"smoker share — train {y_train.mean():.3f} | test {y_test.mean():.3f}")

train 43,547 rows | test 10,887 rows
smoker share — train 0.194 | test 0.194


## 6. The baseline to beat

Every model below is compared against a model that always answers "does not
smoke". This is the number that matters: anything that fails to beat it has
learned nothing.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


def evaluate(name, model):
    """Score a fitted model on the held-out test set."""
    predictions = model.predict(X_test)
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_smoker": precision_score(y_test, predictions, zero_division=0),
        "recall_smoker": recall_score(y_test, predictions, zero_division=0),
        "f1_smoker": f1_score(y_test, predictions, zero_division=0),
    }


results = []

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print(classification_report(y_test, baseline_predictions, zero_division=0))
print(confusion_matrix(y_test, baseline_predictions))

results.append(evaluate("Baseline (always 'no')", baseline))

Accuracy: 0.806374575181409
              precision    recall  f1-score   support

           0       0.81      1.00      0.89      8779
           1       0.00      0.00      0.00      2108

    accuracy                           0.81     10887
   macro avg       0.40      0.50      0.45     10887
weighted avg       0.65      0.81      0.72     10887

[[8779    0]
 [2108    0]]


## 7. Logistic regression: the accuracy trap

The first model looks strong and is not. Its accuracy sits within a fraction of a
point of the baseline, because it has found the cheapest way to be right most of
the time: predict "does not smoke" almost always.

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append(evaluate("Logistic Regression", model))

Accuracy: 0.8199687700927712

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.97      0.90      8779
           1       0.62      0.18      0.28      2108

    accuracy                           0.82     10887
   macro avg       0.73      0.58      0.59     10887
weighted avg       0.79      0.82      0.78     10887


Confusion Matrix:
[[8540  239]
 [1721  387]]


The confusion matrix gives it away: the columns for "smoker" are almost empty.

Re-fitting with `class_weight="balanced"` tells the model that the two classes
cost the same to get wrong. Accuracy goes **down** and the model gets **better** —
the point of the whole exercise.

In [13]:
model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

results.append(evaluate("Logistic Regression (balanced)", model_balanced))

Accuracy: 0.7091944520988335

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.74      0.80      8779
           1       0.35      0.59      0.44      2108

    accuracy                           0.71     10887
   macro avg       0.62      0.66      0.62     10887
weighted avg       0.78      0.71      0.73     10887


Confusion Matrix:
[[6473 2306]
 [ 860 1248]]


The swap is the point of the whole exercise, and the two confusion matrices are
worth reading side by side.

| | Accuracy | Precision (smoker) | Recall (smoker) | F1 (smoker) |
|---|---|---|---|---|
| Unweighted | 0.820 | 0.618 | 0.184 | 0.283 |
| Balanced | 0.709 | 0.351 | 0.592 | 0.441 |

Unweighted, the model flags only 626 profiles as smokers and is right about 62% of
the time — but it misses 1,721 of the 2,108 actual smokers. Balanced, it catches
1,248 of them and pays for that with 2,306 false positives.

Accuracy fell by eleven points and the model became materially more useful. Only
precision got worse, and only because "who should I message?" is a different
question from "does this person smoke at all?".

## 8. Which features drive the prediction?

Logistic regression coefficients are readable directly: positive means "more
likely to smoke". Both the unweighted and balanced models are shown side by side
so it is clear the conclusion does not depend on the weighting.

In [14]:
coef_df = pd.DataFrame(
    {
        "feature": X.columns,
        "lr": model.coef_[0],
        "lr_balanced": model_balanced.coef_[0],
    }
)

coef_df.sort_values(by="lr_balanced", ascending=False)

,feature,lr,lr_balanced
1,drugs_often,2.096129,2.074829
2,drugs_sometimes,1.358819,1.377666
18,drinks_very often,0.941054,1.020849
15,drinks_often,0.515386,0.592816
13,body_type_used up,0.364069,0.356827
7,body_type_full figured,0.296144,0.332136
9,body_type_overweight,0.066634,0.108141
4,body_type_average,0.085044,0.079375
11,body_type_skinny,0.070490,0.042403
5,body_type_curvy,0.063853,0.038809


## 9. Random forest

A forest should beat a linear model if the relationship is not additive. Left at
its default depth it will happily memorise the training set, so the depth is
controlled next.

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))

results.append(evaluate("Random Forest (unlimited depth)", rf))

              precision    recall  f1-score   support

           0       0.87      0.76      0.81      8779
           1       0.35      0.53      0.42      2108

    accuracy                           0.71     10887
   macro avg       0.61      0.65      0.61     10887
weighted avg       0.77      0.71      0.73     10887



In [16]:
# Capping the depth stops the trees from fitting noise. Compare this row against
# the unlimited-depth forest above.
rf5 = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight="balanced", random_state=42, n_jobs=-1
)
rf5.fit(X_train, y_train)

pred5 = rf5.predict(X_test)
print(classification_report(y_test, pred5, zero_division=0))

results.append(evaluate("Random Forest (max_depth=5)", rf5))

              precision    recall  f1-score   support

           0       0.87      0.81      0.84      8779
           1       0.39      0.52      0.45      2108

    accuracy                           0.75     10887
   macro avg       0.63      0.66      0.64     10887
weighted avg       0.78      0.75      0.76     10887



## 10. Tuning tree depth

One depth is not obviously right, so sweep a few and look at the whole table rather than one score.

In [17]:
rows = []
for depth in [3, 5, 7, 10, 15, None]:
    forest = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    forest.fit(X_train, y_train)
    predictions = forest.predict(X_test)

    rows.append(
        {
            "max_depth": "unlimited" if depth is None else depth,
            "accuracy": accuracy_score(y_test, predictions),
            "precision_smoker": precision_score(y_test, predictions, zero_division=0),
            "recall_smoker": recall_score(y_test, predictions, zero_division=0),
            "f1_smoker": f1_score(y_test, predictions, zero_division=0),
        }
    )

sweep = pd.DataFrame(rows)
sweep

,max_depth,accuracy,precision_smoker,recall_smoker,f1_smoker
0,3,0.752182,0.391703,0.506167,0.441639
1,5,0.750712,0.391708,0.519924,0.446800
2,7,0.751630,0.394476,0.528463,0.451744
3,10,0.740975,0.382663,0.550759,0.451575
4,15,0.731331,0.366197,0.530361,0.433249
5,unlimited,0.713787,0.345494,0.534630,0.419739


An earlier version of this analysis concluded that `max_depth=7` was best because it
had "the highest recall, accuracy and f1_score". With the numbers in front of us,
that is not right:

- the highest **accuracy** is at depth 3 (0.752), with depth 7 at 0.752 — a gap of
  0.0006, which is nothing;
- the highest **recall** is at depth 10 (0.551), comfortably clear of depth 7 (0.528);
- the highest **F1** is at depth 7 (0.4517), but depth 10 is 0.4516.

The honest reading is that **anything from depth 3 to depth 10 performs about the
same, and every one of them beats an uncapped tree.** Those depths span 0.010 of F1,
which is well inside the fold-to-fold variation measured in the next section, so
crowning a winner from this table alone is reading noise.

What the table *does* establish is that letting the trees grow without limit is
worse: accuracy 0.714 and F1 0.420, the lowest of the six rows. That is the
overfitting the cap exists to prevent, and it is the one conclusion this sweep
genuinely supports.

## 11. Does it hold up under cross-validation?

Picking a depth from a single test set invites reading noise as signal. Five-fold
stratified cross-validation refits on four folds and scores on the held-out fold,
five times over, which shows how stable each setting actually is.

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for depth in [3, 5, 7, None]:
    forest = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    scores = cross_val_score(forest, X, y, cv=cv, scoring="f1")
    cv_rows.append(
        {
            "max_depth": "unlimited" if depth is None else depth,
            "f1_mean": scores.mean(),
            "f1_std": scores.std(),
            "f1_min": scores.min(),
            "f1_max": scores.max(),
        }
    )

pd.DataFrame(cv_rows)

,max_depth,f1_mean,f1_std,f1_min,f1_max
0,3,0.440864,0.006943,0.432021,0.448485
1,5,0.443438,0.004537,0.437977,0.449200
2,7,0.443875,0.005174,0.437228,0.450915
3,unlimited,0.406895,0.005575,0.401736,0.417394


Cross-validation confirms both halves of that reading.

Depths 3, 5 and 7 land at 0.4409, 0.4434 and 0.4439 mean F1 — a spread of 0.003,
against fold-to-fold standard deviations of about 0.005. They are indistinguishable.
An uncapped forest falls to 0.4069, roughly seven standard deviations below the
capped versions, which is the one difference here that is unambiguously real.

So the defensible claim is not "depth 7 is best" but **"cap the depth; the exact
value barely matters."** Treating 0.4439 as a victory over 0.4434 would have been a
mistake that a single test-set number could not have exposed.

## 12. Does religion add anything?

An earlier version of this analysis carried religion into the feature set and then
dropped it, which left the claim "it barely matters" without any numbers behind it.
This section runs the comparison properly: the same forest, the same folds, the
only difference being whether religion is present.

In [19]:
religion_features = ["drugs", "body_type", "age", "drinks", "religion_main"]

X_religion = pd.get_dummies(
    df[religion_features],
    columns=["drugs", "body_type", "drinks", "religion_main"],
    drop_first=True,
)

X_rel_train, X_rel_test, y_rel_train, y_rel_test = train_test_split(
    X_religion, y, test_size=0.2, random_state=42, stratify=y
)

# The two matrices come from df in the same row order and are split with the same
# seed, so the targets must line up with the model above -- asserted, not assumed.
assert y_rel_train.equals(y_train) and y_rel_test.equals(y_test)

rows = []
for label, (train, test) in {
    "without religion": (X_train, X_test),
    "with religion": (X_rel_train, X_rel_test),
}.items():
    forest = RandomForestClassifier(
        n_estimators=200, max_depth=7, class_weight="balanced", random_state=42, n_jobs=-1
    )
    forest.fit(train, y_train)
    predictions = forest.predict(test)

    rows.append(
        {
            "feature set": label,
            "columns": train.shape[1],
            "accuracy": accuracy_score(y_test, predictions),
            "recall_smoker": recall_score(y_test, predictions, zero_division=0),
            "f1_smoker": f1_score(y_test, predictions, zero_division=0),
        }
    )

pd.DataFrame(rows)

,feature set,columns,accuracy,recall_smoker,f1_smoker
0,without religion,19,0.751630,0.528463,0.451744
1,with religion,27,0.744007,0.538425,0.448883


Religion does not earn its columns.

| Feature set | Columns | Accuracy | Recall (smoker) | F1 (smoker) |
|---|---|---|---|---|
| Without religion | 19 | 0.752 | 0.528 | 0.452 |
| With religion | 27 | 0.744 | 0.538 | 0.449 |

Adding `religion_main` costs eight extra columns and nudges accuracy *down* by 0.008
and F1 down by 0.003, while buying 0.010 of recall. All of that sits inside the noise
band established above. The earlier claim that religion "contributed very little" was
directionally right, but it only means something now that there are numbers attached
to it — and the accurate framing is that it made the model no better, not that it made
no difference at all.

## 13. Results summary

Every model in one place, on the same held-out test set, sorted by smoker F1.

In [20]:
summary = pd.DataFrame(results)[
    ["model", "accuracy", "precision_smoker", "recall_smoker", "f1_smoker"]
]
summary.sort_values("f1_smoker", ascending=False).round(3)

,model,accuracy,precision_smoker,recall_smoker,f1_smoker
4,Random Forest (max_depth=5),0.751,0.392,0.520,0.447
2,Logistic Regression (balanced),0.709,0.351,0.592,0.441
3,Random Forest (unlimited depth),0.714,0.345,0.535,0.420
1,Logistic Regression,0.820,0.618,0.184,0.283
0,Baseline (always 'no'),0.806,0.000,0.000,0.000


## 14. Limitations and what I would do next

### What the model is actually learning

The strongest predictors by coefficient are `drugs_often` (2.07), `drugs_sometimes`
(1.38), `drinks_very often` (1.02) and `drinks_often` (0.59). `age` sits at -0.04 and
no body type exceeds 0.36.

That is close to tautological. Drug use and drinking are not independent evidence
about smoking — they are the same self-reported lifestyle cluster, answered by the
same person, in the same session, about the same habits. A model predicting "smokes"
from "uses drugs" is largely recovering a correlation that was already sitting in the
response pattern. It would be a mistake to present it as a discovery about people.

### Limitations

- **The missing-data hole.** 11,317 rows have no drug answer and are encoded exactly
  like "never". Any imputation decision here would move the results, and the current
  encoding quietly assumes the least informative option.
- **Precision on the minority class stays low.** The best F1 is 0.45, so roughly three
  in five profiles flagged as smokers are not. Useful for prioritising attention, not
  for concluding anything about an individual.
- **The text is ignored.** Ten columns of free-text essays were never touched. If
  anything in this dataset carries signal beyond the structured fields, it is there.
- **One dataset, one year, one city.** These are Bay Area profiles from 2012, and
  smoking prevalence has fallen since. The coefficients should not be read as current.

### What I would do next

1. Give "did not answer" its own explicit level instead of silently merging it with
   "never", then measure how far the coefficients move.
2. Add essay text as features, since the structured columns have clearly been mined out.
3. Report precision-recall curves and choose an operating point deliberately, rather
   than picking between the unweighted and balanced models after the fact.
4. Drop `body_type` and confirm under cross-validation that nothing is lost.

### The honest summary

Smoking is weakly predictable from the rest of an OkCupid profile, and most of the
predictability comes from habits already known to travel together. The genuinely useful
outcome of this exercise was not a model. It was establishing that the 82% accuracy
which looked like success was the base rate wearing a costume, and that fixing it makes
the headline number worse while making the model better.